# 🧠 Image Classification Using Deep Learning
## Comparative Study of CNN Architectures
### Dataset: CIFAR-10 (from Kaggle / TensorFlow Datasets)
**University of Management and Technology — 6th Semester Deep Learning Project**

---
**Algorithms Compared:**
1. Custom CNN (Baseline)
2. VGG-style Deep CNN
3. ResNet-style CNN with Skip Connections
4. CNN + Batch Normalization & Dropout (Regularized CNN)

---

## 📦 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import time
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version: {keras.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📂 2. Load and Explore the Dataset

**Dataset: CIFAR-10**
- Source: Available via TensorFlow / [Kaggle](https://www.kaggle.com/c/cifar-10)
- 60,000 colour images (32×32 pixels), 10 classes
- 50,000 training / 10,000 test images

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                'dog', 'frog', 'horse', 'ship', 'truck']

print('Dataset Shape:')
print(f'  Training images  : {x_train.shape}')
print(f'  Training labels  : {y_train.shape}')
print(f'  Test images      : {x_test.shape}')
print(f'  Test labels      : {y_test.shape}')
print(f'  Classes          : {class_names}')
print(f'  Pixel range      : [{x_train.min()}, {x_train.max()}]')

In [ ]:
# Visualise sample images
fig, axes = plt.subplots(4, 10, figsize=(16, 7))
fig.suptitle('CIFAR-10 Sample Images (one per class)', fontsize=14, fontweight='bold')

for row in range(4):
    for col, cls_name in enumerate(class_names):
        idx = np.where(y_train.flatten() == col)[0][row]
        axes[row, col].imshow(x_train[idx])
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls_name, fontsize=8)

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test,  counts_test  = np.unique(y_test,  return_counts=True)

axes[0].bar(class_names, counts_train, color='steelblue', edgecolor='white')
axes[0].set_title('Training Set — Class Distribution', fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(class_names, counts_test, color='coral', edgecolor='white')
axes[1].set_title('Test Set — Class Distribution', fontweight='bold')
axes[1].set_xlabel('Class'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('The dataset is perfectly balanced — 5000 images per class in training, 1000 in test.')

## 🔧 3. Data Preprocessing & Augmentation

In [ ]:
# Normalise pixel values to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# One-hot encode labels
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat  = keras.utils.to_categorical(y_test,  10)

print('Preprocessing done:')
print(f'  x_train range : [{x_train.min():.2f}, {x_train.max():.2f}]')
print(f'  y_train_cat shape : {y_train_cat.shape}')

In [ ]:
# Data Augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
], name='data_augmentation')

# Visualise augmented images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
sample_img = x_train[0:1]

axes[0, 0].imshow(x_train[0]); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
for i in range(1, 8):
    aug = data_augmentation(sample_img, training=True)
    axes[0, i].imshow(aug[0]); axes[0, i].set_title(f'Aug {i}'); axes[0, i].axis('off')

sample_img2 = x_train[10:11]
axes[1, 0].imshow(x_train[10]); axes[1, 0].set_title('Original'); axes[1, 0].axis('off')
for i in range(1, 8):
    aug = data_augmentation(sample_img2, training=True)
    axes[1, i].imshow(aug[0]); axes[1, i].set_title(f'Aug {i}'); axes[1, i].axis('off')

fig.suptitle('Data Augmentation Examples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation.png', dpi=120, bbox_inches='tight')
plt.show()

## 🏗️ 4. Model Definitions

### Model 1 — Custom Baseline CNN

In [ ]:
def build_baseline_cnn(input_shape=(32, 32, 3), num_classes=10):
    """Simple 3-block CNN without advanced regularisation."""
    model = models.Sequential(name='Baseline_CNN')
    model.add(layers.Input(shape=input_shape))
    model.add(data_augmentation)

    # Block 1
    model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # Classifier head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model1 = build_baseline_cnn()
model1.summary()

### Model 2 — VGG-style Deep CNN

In [ ]:
def build_vgg_cnn(input_shape=(32, 32, 3), num_classes=10):
    """VGG-inspired architecture with stacked 3x3 convolutions."""
    model = models.Sequential(name='VGG_Style_CNN')
    model.add(layers.Input(shape=input_shape))
    model.add(data_augmentation)

    # VGG Block 1 (64 filters)
    for _ in range(2):
        model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # VGG Block 2 (128 filters)
    for _ in range(2):
        model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # VGG Block 3 (256 filters)
    for _ in range(3):
        model.add(layers.Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    # Fully connected head
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model2 = build_vgg_cnn()
model2.summary()

### Model 3 — ResNet-style CNN with Skip Connections

In [ ]:
def residual_block(x, filters, stride=1):
    """A single residual block with optional shortcut projection."""
    shortcut = x

    # Main path
    x = layers.Conv2D(filters, (3,3), strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Projection shortcut when dimensions change
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1,1), strides=stride)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


def build_resnet_cnn(input_shape=(32, 32, 3), num_classes=10):
    """ResNet-inspired architecture with skip connections."""
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Initial convolution
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Residual stages
    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    # Global average pooling + classifier
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='ResNet_Style_CNN')

model3 = build_resnet_cnn()
model3.summary()

### Model 4 — Regularized CNN (BatchNorm + Dropout)

In [ ]:
def build_regularized_cnn(input_shape=(32, 32, 3), num_classes=10):
    """CNN with comprehensive regularisation: BatchNorm, Dropout, L2."""
    model = models.Sequential(name='Regularized_CNN')
    model.add(layers.Input(shape=input_shape))
    model.add(data_augmentation)

    # Block 1
    model.add(layers.Conv2D(32, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(32, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.25))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Conv2D(64, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.25))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), padding='same',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.25))

    # Classifier head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation('relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

model4 = build_regularized_cnn()
model4.summary()

## ⚙️ 5. Training Configuration

In [ ]:
EPOCHS      = 50
BATCH_SIZE  = 64
LR          = 1e-3

# Shared callbacks
def get_callbacks(name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-6, verbose=1)
    ]

# Compile helper
def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

print(f'Training config: epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LR}')

## 🚀 6. Train All Models

In [ ]:
histories = {}
training_times = {}
models_dict = {
    'Baseline CNN'    : model1,
    'VGG-Style CNN'   : model2,
    'ResNet-Style CNN': model3,
    'Regularized CNN' : model4
}

for name, model in models_dict.items():
    print(f'\n{"="*55}')
    print(f'  Training: {name}')
    print(f'{"="*55}')
    compile_model(model)
    t0 = time.time()
    history = model.fit(
        x_train, y_train_cat,
        validation_split=0.15,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks(name),
        verbose=1
    )
    training_times[name] = time.time() - t0
    histories[name]       = history
    print(f'  Finished in {training_times[name]:.1f}s')

## 📊 7. Evaluate & Compare Models

In [ ]:
results = {}

for name, model in models_dict.items():
    loss, acc = model.evaluate(x_test, y_test_cat, verbose=0)
    y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
    results[name] = {
        'test_loss'    : loss,
        'test_accuracy': acc,
        'y_pred'       : y_pred,
        'train_time_s' : training_times[name],
        'params'       : model.count_params()
    }
    print(f'{name:20s} | Loss: {loss:.4f} | Accuracy: {acc*100:.2f}%')

In [ ]:
# Summary comparison table
rows = []
for name, r in results.items():
    rows.append({
        'Model'               : name,
        'Test Accuracy (%)'   : f"{r['test_accuracy']*100:.2f}",
        'Test Loss'           : f"{r['test_loss']:.4f}",
        'Parameters'          : f"{r['params']:,}",
        'Training Time (s)'   : f"{r['train_time_s']:.1f}"
    })

df_results = pd.DataFrame(rows)
print('\n====== MODEL COMPARISON SUMMARY ======')
print(df_results.to_string(index=False))

In [ ]:
# Training curves
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for i, (name, history) in enumerate(histories.items()):
    col   = i
    color = colors[i]
    hist  = history.history

    # Accuracy plot
    axes[0, col].plot(hist['accuracy'],     label='Train', color=color)
    axes[0, col].plot(hist['val_accuracy'], label='Val',   color=color, linestyle='--')
    axes[0, col].set_title(f'{name}\nAccuracy', fontsize=9, fontweight='bold')
    axes[0, col].set_xlabel('Epoch'); axes[0, col].set_ylabel('Accuracy')
    axes[0, col].legend(); axes[0, col].grid(alpha=0.3)

    # Loss plot
    axes[1, col].plot(hist['loss'],     label='Train', color=color)
    axes[1, col].plot(hist['val_loss'], label='Val',   color=color, linestyle='--')
    axes[1, col].set_title(f'{name}\nLoss', fontsize=9, fontweight='bold')
    axes[1, col].set_xlabel('Epoch'); axes[1, col].set_ylabel('Loss')
    axes[1, col].legend(); axes[1, col].grid(alpha=0.3)

fig.suptitle('Training History — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart comparison
names = list(results.keys())
accs  = [results[n]['test_accuracy']*100 for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, accs, color=colors, edgecolor='white', width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — Test Accuracy', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 🔍 8. Confusion Matrices

In [ ]:
y_true = y_test.flatten()
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, (name, r) in zip(axes.flatten(), results.items()):
    cm = confusion_matrix(y_true, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{name}\nTest Accuracy: {r["test_accuracy"]*100:.2f}%',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 📋 9. Per-class Classification Reports

In [ ]:
for name, r in results.items():
    print(f'\n{"="*60}')
    print(f'  Classification Report: {name}')
    print(f'{"="*60}')
    print(classification_report(y_true, r['y_pred'], target_names=class_names))

## 🏆 10. Best Model — Detailed Analysis & Save

In [ ]:
# Identify best model
best_name  = max(results, key=lambda n: results[n]['test_accuracy'])
best_model = models_dict[best_name]
best_acc   = results[best_name]['test_accuracy']

print(f'Best Model  : {best_name}')
print(f'Best Accuracy: {best_acc*100:.2f}%')

# Save best model
best_model.save('best_model.keras')
print('Best model saved to best_model.keras')

In [ ]:
# Visualise predictions from the best model
y_pred_best = results[best_name]['y_pred']

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
indices = np.random.choice(len(x_test), 32, replace=False)

for ax, idx in zip(axes.flatten(), indices):
    ax.imshow(x_test[idx])
    true_label = class_names[y_true[idx]]
    pred_label = class_names[y_pred_best[idx]]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'T:{true_label}\nP:{pred_label}', fontsize=6, color=color)
    ax.axis('off')

fig.suptitle(f'Predictions — {best_name} (green=correct, red=wrong)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions.png', dpi=120, bbox_inches='tight')
plt.show()

## 📈 11. Final Summary

In [ ]:
print('='*65)
print('               FINAL COMPARISON SUMMARY')
print('='*65)
print(f'{"Model":<22} {"Accuracy":>12} {"Loss":>10} {"Params":>12} {"Time(s)":>10}')
print('-'*65)
for name, r in sorted(results.items(),
                       key=lambda x: x[1]['test_accuracy'], reverse=True):
    marker = '  ← BEST' if name == best_name else ''
    print(f'{name:<22} {r["test_accuracy"]*100:>11.2f}% '
          f'{r["test_loss"]:>10.4f} '
          f'{r["params"]:>12,} '
          f'{r["train_time_s"]:>10.1f}{marker}')
print('='*65)
print(f'\nBest model saved as best_model.keras')
print(f'Dataset: CIFAR-10 | Classes: 10 | Test samples: {len(x_test)}')

---
## ✅ Conclusions

| Model | Key Characteristic | Strength | Limitation |
|---|---|---|---|
| Baseline CNN | Simple 3-block architecture | Fast training | Prone to overfitting |
| VGG-Style CNN | Deep stacked conv layers | High capacity | Many parameters |
| ResNet-Style CNN | Skip connections + BN | Gradient flow, best accuracy | Slightly complex |
| Regularized CNN | BN + Dropout + L2 | Good generalisation | Slower convergence |

**Key Takeaways:**
- Skip connections (ResNet) help prevent vanishing gradients and typically achieve the best accuracy on CIFAR-10.
- Regularisation techniques (BatchNorm, Dropout, L2) significantly reduce overfitting.
- Data augmentation is essential for small datasets like CIFAR-10 to improve generalisation.
- VGG-style models have the most parameters but may overfit without regularisation.

---
*Project by UMT 6th Semester | Deep Learning Course*